In [ ]:
# Load all necessary packages
import sys
sys.path.insert(1, "../")  

import numpy as np
np.random.seed(0)

from aif360.datasets import GermanDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing

from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from fairlearn.preprocessing import CorrelationRemover
import pandas as pd
from sklearn.datasets import fetch_openml

### L4: Data Mitigation Demo

Example taken from AIF360 tutorials

In [ ]:
#Load data

label_map = {1.0: 'Good Credit', 2.0: 'Bad Credit'}
protected_attribute_maps = [{1.0: 'Male', 0.0: 'Female'}]

gd = GermanDataset(protected_attribute_names=['sex'],
privileged_classes=[['male']], metadata={'label_map': label_map,
'protected_attribute_maps': protected_attribute_maps})

df, df_dict = gd.convert_to_dataframe()

In [ ]:
df.head()

In [ ]:
# Metric for the original dataset
privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]

dataset_orig_train, dataset_orig_test = gd.split([0.7], shuffle=True)

metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
print("Statistical Parity Difference between unprivileged and privileged groups = %f" % metric_orig_train.mean_difference())
print("Disparate Impact between unprivileged and privileged groups = %f" % metric_orig_train.disparate_impact())

Interpretation:

**Statistical Parity Difference: ** The metric is less than 0, indicating higher benefit for the privilege group  - men.

**Disparate Impact: ** The ideal value for this metric is a 1.0. 
The metric is less than 1, indicating a higher benefit for the privileged group, men.

In [ ]:
dataset_orig_train.instance_weights

In [ ]:
#Apply the reweighing algorithm to reweigh labels
RW = Reweighing(unprivileged_groups=unprivileged_groups,
               privileged_groups=privileged_groups)
RW.fit(dataset_orig_train)
dataset_transf_train = RW.transform(dataset_orig_train)

In [ ]:
dataset_transf_train_df, dataset_transf_dict = dataset_transf_train.convert_to_dataframe()

In [ ]:
dataset_transf_train_df

In [ ]:
dataset_transf_train.instance_weights

Instance weights that are < 1 likely correspond to those with unfavorable outcomes, bad credit, for the unprivileged group, women, and favorable (good credit) outcomes for the privileged group (men).

Instance weights are > 1 likely correspond to  favorable outcomes for the unprivileged group and unfavorable outcomes for the privileged group.

In [ ]:
metric_orig_train = BinaryLabelDatasetMetric(dataset_transf_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
print("Statistical Parity Difference between unprivileged and privileged groups = %f" % metric_orig_train.mean_difference())
print("Disparate Impact between unprivileged and privileged groups = %f" % metric_orig_train.disparate_impact())

Interpretation:

**Statistical Parity Difference: ** If the reported metric is 0, this indicates equal benefit for both groups. 

**Disparate Impact: ** The ideal value for this metric is a 1.0. 

**Conclusion:** Bias was succesfuly mitigated using the reweighing algorithm.

# Correlation Remover

Example taken from: https://fairlearn.org/main/user_guide/mitigation.html

In [ ]:
def plot_heatmap(df, title):
    df['target'] = y
    df = df.rename(columns={"had_inpatient_days_True": "had_inpatient_days"})
    cols = list(df.columns)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(round(df.corr(), 2), cmap="coolwarm")

    # Show all ticks and label them with the respective list entries
    plt.setp(ax, xticks=np.arange(len(cols)), xticklabels=cols,
        yticks=np.arange(len(cols)), yticklabels=cols)

    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right", rotation_mode="anchor")

    # Loop over data dimensions and create text annotations.
    for i in range(len(cols)):
        for j in range(len(cols)):
            ax.text(
                j,
                i,
                round(df.corr().to_numpy()[i, j], 2),
                ha="center",
                va="center",
            )

    ax.set_title(f"{title}")
    plt.show()

In [ ]:
data = fetch_openml(data_id=43874, as_frame=True)
X_raw = data.data[["race", "time_in_hospital", "had_inpatient_days", "medicare"]]
X_raw = pd.get_dummies(X_raw)
y = data.target

X_raw = X_raw.drop(["race_Asian",
                    'race_Caucasian',
                    'race_Hispanic',
                    'race_Other',
                    'race_Unknown',
                    'had_inpatient_days_False',
                    'medicare_False'], axis=1)

X_raw = X_raw[['time_in_hospital', 'had_inpatient_days_True', 'medicare_True', 'race_AfricanAmerican']]

In [ ]:
plot_heatmap(X_raw, "Correlation values in the original dataset")

*Interpretation:* In the first plot, we see limited correlation between ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Â¹Ãƒâ€¦Ã¢â‚¬Å“race_AfricanAmericanÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¾Ãƒâ€šÃ‚Â¢ and the other columns.

In [ ]:
cr = CorrelationRemover(sensitive_feature_ids=["race_AfricanAmerican"])
X_cr = cr.fit_transform(X_raw)
X_cr = pd.DataFrame(X_cr, columns=['time_in_hospital', 'had_inpatient_days_True', 'medicare_True', 'race_AfricanAmerican'])
X_cr["race_AfricanAmerican"] = X_raw["race_AfricanAmerican"]

In [ ]:
plot_heatmap(X_cr, "Correlation values after CorrelationRemover")

*Interpretation:* Correlation Remover has succesfully removed these correlations.

# Data Techniques for Enhanced Fairness

In this exercise, you will implement the Reweighing and Correlation Remover strategies for the *Diabetes* dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from fairlearn.preprocessing import CorrelationRemover
from aif360.datasets import StandardDataset, BinaryLabelDataset
from aif360.metrics import ClassificationMetric, BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing

In [ ]:
#Dataset loading

data = fetch_openml(data_id=43874, as_frame=True)
X_raw = data.data[["race", "time_in_hospital", "had_inpatient_days", "medicare"]]
X_raw = pd.get_dummies(X_raw)
y = data.target

X_raw = X_raw.drop(["race_Asian",
                    'race_AfricanAmerican',
                    'race_Hispanic',
                    'race_Other',
                    'race_Unknown',
                    'had_inpatient_days_False',
                    'medicare_False'], axis=1)

#Include the Caucasian variable as part of the analysis
X_raw = X_raw[['time_in_hospital', 'had_inpatient_days_True', 'medicare_True', 'race_Caucasian']]

### Correlation Remover

In this section, you'll use the Correlation Remover to inspect mitigate potential correlations for the Caucasian group.

In [ ]:
##Helper code provided by Fairlearn tutorials
def plot_heatmap(df, title):
    df['target'] = y
    df = df.rename(columns={"had_inpatient_days_True": "had_inpatient_days"})
    cols = list(df.columns)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(round(df.corr(), 2), cmap="coolwarm")

    # Show all ticks and label them with the respective list entries
    ax.set_xticks(np.arange(len(cols)))
    ax.set_xticklabels(cols)
    ax.set_yticks(np.arange(len(cols)))
    ax.set_yticklabels(cols)

    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=15, ha="right", rotation_mode="anchor")

    # Loop over data dimensions and create text annotations.
    for i in range(len(cols)):
        for j in range(len(cols)):
            ax.text(
                j,
                i,
                round(df.corr().to_numpy()[i, j], 2),
                ha="center",
                va="center",
            )

    ax.set_title(f"{title}")
    plt.show()

Below, we plot the correlation values in the original dataset.

In [ ]:
plot_heatmap(X_raw, "Correlation values in the original dataset")

Inspect the correlation matrix. What types of correlations is the Caucasian variable exhibiting?

*Interpretation: The Caucasian variable has a low level of correlation with all the other variables. It has a slightly positive correlation with the target and other variables, and a slight negative correlation with the time_in_hopsital variable.*

Use the Correlation Remover method to transform the dataset. Hint: For the usage, refer to the demo and the library tutorial for an example.

In [ ]:
### YOUR CODE HERE
cr = CorrelationRemover(sensitive_feature_ids=["race_Caucasian"])
X_cr = cr.fit_transform(X_raw)

In [ ]:
X_cr = pd.DataFrame(X_cr, columns=['time_in_hospital', 'had_inpatient_days_True', 'medicare_True', 'race_Caucasian'])
X_cr["race_Caucasian"] = X_raw["race_Caucasian"]

Plot the correlation heatmap after applying the bias mitigation method.

In [ ]:
plot_heatmap(X_cr, "Correlation values after CorrelationRemover")

Interpret your findings.

*Interpretation: After applying the Correlation Remover algorithm with its defaults, we see correlations between the race_Caucasian variable and other variables are now 0. The exception is that there is still a slight positive correlation between the Caucasian and target variable.*

### Reweighing

Let's now apply the reweighing algorithm on the dataset.

In [ ]:
X_raw['readmit_30_days'] = y

Let's use the BinaryLabelDataset from the AIF360 package. 

For this analysis, let's present a hypothetical scenario **where individuals who do not identify as Caucasian are priviledged**, and individuals who **do identify as Caucasian are unprivileged**.

Define the inputs to the BinaryLabelDataset call.

1. What is the favorable label, in terms of the variable we're trying to predict? Hint: readmitting to the hopsital is **not** favorable.
2. Who is the privileged vs. unprivileged group?

In [ ]:
#YOUR CODE HERE
favorable_label = 0 #Not being re-admitted
unfavorable_label = 1 #Being re-admitted

privileged_groups = [{'race_Caucasian': 0}] #Not being Caucasian
unprivileged_groups = [{'race_Caucasian': 1}] #Being Caucasian

In [ ]:
binary_act_dataset = BinaryLabelDataset(df=X_raw, 
                                  label_names=['readmit_30_days'],
                                  protected_attribute_names=['race_Caucasian'],
                                  favorable_label=favorable_label, 
                                  unfavorable_label=unfavorable_label) 

Next, we'll look at the statistical parity and disparate impact metrics. We've included helper code below to generate these metrics. Interpret these values.

Hint: Remember that we defined **individuals who do not identify as Caucasian are priviledged**, and individuals who **do identify as Caucasian are unprivileged**.


In [ ]:
orig_metric_act_dataset = BinaryLabelDatasetMetric(binary_act_dataset, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Statistical Parity Difference = %f" % orig_metric_act_dataset.mean_difference())
print(f"Disparate impact: {orig_metric_act_dataset.disparate_impact():.4f}")

*Interpretation:* 

For statistical parity, we see the metric is close, but greater than 0, indicating a higher benefit for the **unprivileged** group.

For disparate impact, we see the metric is greater than one 1, also implying a higher benefit for the **unprivileged** group.

This indicates that individuals who **are** Caucasian have a higher benefit over individuals that are not Caucasian. 

Next, let's apply the reweighing algorithm to your dataset using the RW.fit_transform() function to see if we can mitigate the bias. Hint: For the usage, refer to the demo and the library tutorial for an example.

In [ ]:
#YOUR CODE HERE
RW = Reweighing(unprivileged_groups=unprivileged_groups,
                privileged_groups=privileged_groups)
reweighed_data = RW.fit_transform(binary_act_dataset)

Measure the statistical parity difference and disparate impact. Interpret how the values changed.

In [ ]:
new_metric = BinaryLabelDatasetMetric(reweighed_data, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Statistical Parity Difference = %f" % new_metric.mean_difference())
print(f"Disparate impact: {new_metric.disparate_impact():.4f}")

*Interpretation:* 

For statistical parity, we see the metric is at 0, the ideal value for this metric.

For disparate impact, we see the metric is at 1, the ideal value for this metric.

It seems we have succesfully mitigated bias in the dataset for our particular setup using the Reweighing algorithm.

In [ ]:
# Load all necessary packages
import sys
sys.path.insert(1, "../")  

import tensorflow.compat.v1 as tf
import numpy as np
np.random.seed(0)

from aif360.datasets import GermanDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.inprocessing.adversarial_debiasing import AdversarialDebiasing
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve

### Load Dataset

In [ ]:
#Set TF session to None

#Load data
label_map = {1.0: 'Good Credit', 2.0: 'Bad Credit'}
protected_attribute_maps = [{1.0: 'Male', 0.0: 'Female'}]

gd = GermanDataset(protected_attribute_names=['sex'],
privileged_classes=[['male']], metadata={'label_map': label_map,
'protected_attribute_maps': protected_attribute_maps})

dataset_orig_train, dataset_orig_test = gd.split([0.7], shuffle=True)

privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]

In [ ]:
sess = None
scale_orig = StandardScaler()
X_train = scale_orig.fit_transform(dataset_orig_train.features)
y_train = dataset_orig_train.labels.ravel()
w_train = dataset_orig_train.instance_weights.ravel()

lmod = LogisticRegression()
lmod.fit(X_train, y_train, 
         sample_weight=dataset_orig_train.instance_weights)
y_train_pred = lmod.predict(X_train)

In [ ]:
min_max_scaler = MaxAbsScaler()
dataset_orig_train.features = min_max_scaler.fit_transform(dataset_orig_train.features)
dataset_orig_test.features = min_max_scaler.transform(dataset_orig_test.features)
metric_scaled_train = BinaryLabelDatasetMetric(dataset_orig_train, 
                             unprivileged_groups=unprivileged_groups,
                             privileged_groups=privileged_groups)
display(Markdown("#### Scaled dataset - Verify that the scaling does not affect the group label statistics"))
print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_scaled_train.mean_difference())
metric_scaled_test = BinaryLabelDatasetMetric(dataset_orig_test, 
                             unprivileged_groups=unprivileged_groups,
                             privileged_groups=privileged_groups)
print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_scaled_test.mean_difference())

In [ ]:
# Learn parameters with debias set to False
if sess is not None:
  sess.close()
tf.reset_default_graph()
sess = tf.Session()
tf.disable_eager_execution()
plain_model = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='plain_classifier',
                          debias=False,
                          sess=sess)
plain_model.fit(dataset_orig_train)

In [ ]:
# Apply the plain model to test data
dataset_nodebiasing_train = plain_model.predict(dataset_orig_train)
dataset_nodebiasing_test = plain_model.predict(dataset_orig_test)

In [ ]:
# Metrics for the dataset from plain model (without debiasing)
display(Markdown("#### Plain model - without debiasing - dataset metrics"))
metric_dataset_nodebiasing_train = BinaryLabelDatasetMetric(dataset_nodebiasing_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_nodebiasing_train.mean_difference())

metric_dataset_nodebiasing_test = BinaryLabelDatasetMetric(dataset_nodebiasing_test, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_nodebiasing_test.mean_difference())

display(Markdown("#### Plain model - without debiasing - classification metrics"))
classified_metric_nodebiasing_test = ClassificationMetric(dataset_orig_test, 
                                                 dataset_nodebiasing_test,
                                                 unprivileged_groups=unprivileged_groups,
                                                 privileged_groups=privileged_groups)
print("Test set: Classification accuracy = %f" % classified_metric_nodebiasing_test.accuracy())
TPR = classified_metric_nodebiasing_test.true_positive_rate()
TNR = classified_metric_nodebiasing_test.true_negative_rate()
bal_acc_nodebiasing_test = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = %f" % bal_acc_nodebiasing_test)
print("Test set: Disparate impact = %f" % classified_metric_nodebiasing_test.disparate_impact())
print("Test set: Equal opportunity difference = %f" % classified_metric_nodebiasing_test.equal_opportunity_difference())
print("Test set: Average odds difference = %f" % classified_metric_nodebiasing_test.average_odds_difference())
print("Test set: Theil_index = %f" % classified_metric_nodebiasing_test.theil_index())

Interpretation of the model **without debiasing**:

**Equal opportunity difference :** A value > 0 implies higher benefit for the unprivileged group, women.

**Disparate impact:** The ideal value for this metric is a 1.0. 
The metric is close to but less than 1, indiacting a higher benefit for the privileged group, men.

**Average odds difference :** A value of < 0 implies higher benefit for the privileged group, men.

**Theil_index:** A value of 0 implies perfect fairness. Lower scores implies fairness, higher scores are problematic.

-----------------------------------------------------------------

In [ ]:
# Load post-processing algorithm that equalizes the odds
if sess:
  sess.close()
tf.reset_default_graph()
sess = tf.Session()
tf.disable_eager_execution()
# Learn parameters with debias set to True
debiased_model = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='debiased_classifier',
                          debias=True, #setting debias=False enables a plain classifier without debiasing
                          sess=sess)
debiased_model.fit(dataset_orig_train)

In [ ]:
# Apply the plain model to test data
dataset_debiasing_train = debiased_model.predict(dataset_orig_train)
dataset_debiasing_test = debiased_model.predict(dataset_orig_test)

In [ ]:
# Metrics for the dataset from plain model (without debiasing)
display(Markdown("#### Plain model - without debiasing - dataset metrics"))
metric_dataset_nodebiasing_train = BinaryLabelDatasetMetric(dataset_nodebiasing_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_nodebiasing_train.mean_difference())

metric_dataset_nodebiasing_test = BinaryLabelDatasetMetric(dataset_nodebiasing_test, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_nodebiasing_test.mean_difference())

display(Markdown("#### Plain model - without debiasing - classification metrics"))
classified_metric_nodebiasing_test = ClassificationMetric(dataset_orig_test, 
                                                 dataset_nodebiasing_test,
                                                 unprivileged_groups=unprivileged_groups,
                                                 privileged_groups=privileged_groups)
print("Test set: Classification accuracy = %f" % classified_metric_nodebiasing_test.accuracy())
TPR = classified_metric_nodebiasing_test.true_positive_rate()
TNR = classified_metric_nodebiasing_test.true_negative_rate()
bal_acc_nodebiasing_test = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = %f" % bal_acc_nodebiasing_test)
print("Test set: Disparate impact = %f" % classified_metric_nodebiasing_test.disparate_impact())
print("Test set: Equal opportunity difference = %f" % classified_metric_nodebiasing_test.equal_opportunity_difference())
print("Test set: Average odds difference = %f" % classified_metric_nodebiasing_test.average_odds_difference())
print("Test set: Theil_index = %f" % classified_metric_nodebiasing_test.theil_index())

# Metrics for the dataset from model with debiasing
display(Markdown("#### Model - with debiasing - dataset metrics"))
metric_dataset_debiasing_train = BinaryLabelDatasetMetric(dataset_debiasing_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_debiasing_train.mean_difference())

metric_dataset_debiasing_test = BinaryLabelDatasetMetric(dataset_debiasing_test, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_debiasing_test.mean_difference())

display(Markdown("#### Model - with debiasing - classification metrics"))
classified_metric_debiasing_test = ClassificationMetric(dataset_orig_test, 
                                                 dataset_debiasing_test,
                                                 unprivileged_groups=unprivileged_groups,
                                                 privileged_groups=privileged_groups)
print("Test set: Classification accuracy = %f" % classified_metric_debiasing_test.accuracy())
TPR = classified_metric_debiasing_test.true_positive_rate()
TNR = classified_metric_debiasing_test.true_negative_rate()
bal_acc_debiasing_test = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = %f" % bal_acc_debiasing_test)
print("Test set: Disparate impact = %f" % classified_metric_debiasing_test.disparate_impact())
print("Test set: Equal opportunity difference = %f" % classified_metric_debiasing_test.equal_opportunity_difference())
print("Test set: Average odds difference = %f" % classified_metric_debiasing_test.average_odds_difference())
print("Test set: Theil_index = %f" % classified_metric_debiasing_test.theil_index())

Interpretation of the model **with debiasing**:

**Equal opportunity difference :** A value > 0 implies higher benefit for the unprivileged group, women.

**Disparate impact:** The ideal value for this metric is a 1.0. The metric is close to but less than 1, indicating a higher benefit for the privileged group, men. We see an improvement for this metric after debiasing.

**Average odds difference :** A value > 0 implies higher benefit for the privileged group, men.

**Theil_index:** A value of 0 implies perfect fairness. Lower scores implies fairness, higher scores are problematic.

In [ ]:
# Load all necessary packages
import sys
sys.path.insert(1, "../")  

import tensorflow.compat.v1 as tf
import numpy as np
np.random.seed(0)

from aif360.datasets import GermanDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.inprocessing.adversarial_debiasing import AdversarialDebiasing
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve

In [ ]:
#Load data
label_map = {1.0: 'Good Credit', 2.0: 'Bad Credit'}
protected_attribute_maps = [{1.0: 'Male', 0.0: 'Female'}]

gd = GermanDataset(protected_attribute_names=['sex'],
privileged_classes=[['male']], metadata={'label_map': label_map,
'protected_attribute_maps': protected_attribute_maps})

dataset_orig_train, dataset_orig_vt = gd.split([0.7], shuffle=True)
dataset_orig_valid, dataset_orig_test = dataset_orig_vt.split([0.5], shuffle=True)

privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]

In [ ]:
dataset_orig_train, dataset_orig_test = gd.split([0.7], shuffle=True)

metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
print("Statistical Parity Difference between unprivileged and privileged groups = %f" % metric_orig_train.mean_difference())
print("Disparate Impact between unprivileged and privileged groups = %f" % metric_orig_train.disparate_impact())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve

# Placeholder for predicted and transformed datasets
dataset_orig_train_pred = dataset_orig_train.copy(deepcopy=True)
dataset_orig_valid_pred = dataset_orig_valid.copy(deepcopy=True)
dataset_orig_test_pred = dataset_orig_test.copy(deepcopy=True)

dataset_new_valid_pred = dataset_orig_valid.copy(deepcopy=True)
dataset_new_test_pred = dataset_orig_test.copy(deepcopy=True)

# Logistic regression classifier and predictions for training data
scale_orig = StandardScaler()
X_train = scale_orig.fit_transform(dataset_orig_train.features)
y_train = dataset_orig_train.labels.ravel()
lmod = LogisticRegression()
lmod.fit(X_train, y_train)

fav_idx = np.where(lmod.classes_ == dataset_orig_train.favorable_label)[0][0]
y_train_pred_prob = lmod.predict_proba(X_train)[:,fav_idx]

# Prediction probs for validation and testing data
X_valid = scale_orig.transform(dataset_orig_valid.features)
y_valid_pred_prob = lmod.predict_proba(X_valid)[:,fav_idx]

X_test = scale_orig.transform(dataset_orig_test.features)
y_test_pred_prob = lmod.predict_proba(X_test)[:,fav_idx]

class_thresh = 0.5
dataset_orig_train_pred.scores = y_train_pred_prob.reshape(-1,1)
dataset_orig_valid_pred.scores = y_valid_pred_prob.reshape(-1,1)
dataset_orig_test_pred.scores = y_test_pred_prob.reshape(-1,1)

y_train_pred = np.zeros_like(dataset_orig_train_pred.labels)
y_train_pred[y_train_pred_prob >= class_thresh] = dataset_orig_train_pred.favorable_label
y_train_pred[~(y_train_pred_prob >= class_thresh)] = dataset_orig_train_pred.unfavorable_label
dataset_orig_train_pred.labels = y_train_pred

y_valid_pred = np.zeros_like(dataset_orig_valid_pred.labels)
y_valid_pred[y_valid_pred_prob >= class_thresh] = dataset_orig_valid_pred.favorable_label
y_valid_pred[~(y_valid_pred_prob >= class_thresh)] = dataset_orig_valid_pred.unfavorable_label
dataset_orig_valid_pred.labels = y_valid_pred
    
y_test_pred = np.zeros_like(dataset_orig_test_pred.labels)
y_test_pred[y_test_pred_prob >= class_thresh] = dataset_orig_test_pred.favorable_label
y_test_pred[~(y_test_pred_prob >= class_thresh)] = dataset_orig_test_pred.unfavorable_label
dataset_orig_test_pred.labels = y_test_pred

In [ ]:
# Odds equalizing post-processing algorithm
from aif360.algorithms.postprocessing.calibrated_eq_odds_postprocessing import CalibratedEqOddsPostprocessing
from tqdm import tqdm

cost_constraint = "fnr"
randseed = 12345679 

# Learn parameters to equalize odds and apply to create a new dataset
cpp = CalibratedEqOddsPostprocessing(privileged_groups = privileged_groups,
                                     unprivileged_groups = unprivileged_groups,
                                     cost_constraint=cost_constraint,
                                     seed=randseed)
cpp = cpp.fit(dataset_orig_valid, dataset_orig_valid_pred)

dataset_transf_valid_pred = cpp.predict(dataset_orig_valid_pred)
dataset_transf_test_pred = cpp.predict(dataset_orig_test_pred)

In [ ]:
cm_pred_test = ClassificationMetric(dataset_orig_test, dataset_orig_test_pred,
                             unprivileged_groups=unprivileged_groups,
                             privileged_groups=privileged_groups)
display(Markdown("#### Original-Predicted testing dataset"))
print("Difference in GFPR between unprivileged and privileged groups")
print(cm_pred_test.difference(cm_pred_test.generalized_false_positive_rate))
print("Difference in GFNR between unprivileged and privileged groups")
print(cm_pred_test.difference(cm_pred_test.generalized_false_negative_rate))

TPR = cm_pred_test.true_positive_rate()
TNR = cm_pred_test.true_negative_rate()
bal_acc_nodebiasing_test = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = %f" % bal_acc_nodebiasing_test)
print("Test set: Disparate impact = %f" % cm_pred_test.disparate_impact())
print("Test set: Equal opportunity difference = %f" % cm_pred_test.equal_opportunity_difference())
print("Test set: Average odds difference = %f" % cm_pred_test.average_odds_difference())
print("Test set: Theil_index = %f" % cm_pred_test.theil_index())


cm_transf_test = ClassificationMetric(dataset_orig_test, dataset_transf_test_pred,
                             unprivileged_groups=unprivileged_groups,
                             privileged_groups=privileged_groups)
display(Markdown("#### Transformed testing dataset"))
print("Difference in GFPR between unprivileged and privileged groups")
print(cm_transf_test.difference(cm_transf_test.generalized_false_positive_rate))
print("Difference in GFNR between unprivileged and privileged groups")
print(cm_transf_test.difference(cm_transf_test.generalized_false_negative_rate))
TPR = cm_pred_test.true_positive_rate()
TNR = cm_pred_test.true_negative_rate()
bal_acc_nodebiasing_test_transf = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = %f" % bal_acc_nodebiasing_test_transf)
print("Test set: Disparate impact = %f" % cm_transf_test.disparate_impact())
print("Test set: Equal opportunity difference = %f" % cm_transf_test.equal_opportunity_difference())
print("Test set: Average odds difference = %f" % cm_transf_test.average_odds_difference())
print("Test set: Theil_index = %f" % cm_transf_test.theil_index())

Interpretation of the model **after calibrated equalized odds postprocessing**:

**Equal opportunity difference :** The metric is less than 0, indicating higher benefit for the privilege group  - men.

**Disparate impact:** The ideal value for this metric is a 1.0. 
The metric is less than 1, indiacting a higher benefit for the privileged group, men.

**Average odds difference :** A value > 0 implies higher benefit for the privileged group, men. A value < 0 implies higher benefit for the unprivileged group - women.

**Theil_index:** A value of 0 implies perfect fairness. Lower scores implies fairness, higher scores are problematic.

# AI Fairness for Models

In this exercise, you will apply Adversarial Debiasing as an in-processing bias mitigation, and compare the performance of different variations to identify how fairness metrics change if the parameters of Adversarial Debiasing are modified.

In [ ]:
# Load all necessary packages
import sys
sys.path.insert(1, "../")  

import tensorflow.compat.v1 as tf
import numpy as np
np.random.seed(0)

from aif360.datasets import GermanDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.inprocessing.adversarial_debiasing import AdversarialDebiasing
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve

### Load Dataset

In [ ]:
#Load data
label_map = {1.0: 'Good Credit', 2.0: 'Bad Credit'}
protected_attribute_maps = [{1.0: 'Male', 0.0: 'Female'}]

gd = GermanDataset(protected_attribute_names=['sex'],
privileged_classes=[['male']], metadata={'label_map': label_map,
'protected_attribute_maps': protected_attribute_maps})

privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]

## Adversarial Debiasing

Below, we create a simple train-test split of the dataset.

In [ ]:
dataset_orig_train, dataset_orig_test = gd.split([0.7], shuffle=True)

In [ ]:
#Apply scaling
min_max_scaler = MaxAbsScaler()
dataset_orig_train.features = min_max_scaler.fit_transform(dataset_orig_train.features)
dataset_orig_test.features = min_max_scaler.transform(dataset_orig_test.features)

Train an Adversarial Debiasing algorithm with debias set to False, and at least **two** variations of the adversarial debiasing model with debias set to True. 

You can create these variations by modifying the default parameters of the algorithm ([source](https://github.com/Trusted-AI/AIF360/blob/master/aif360/algorithms/inprocessing/adversarial_debiasing.py)):

```
adversary_loss_weight (float, optional): Hyperparameter that chooses the strength of the adversarial loss.
num_epochs (int, optional): Number of training epochs.
batch_size (int, optional): Batch size.
classifier_num_hidden_units (int, optional): Number of hidden units in the classifier model.
```

Measure the balanced accuracy, equal odds opportunity, and disparate impact for each model you create on the test dataset. How susceptible are the fairness metrics to change when you modify the parameters of the algorithm?

For example, you may create and train:

- Adversarial Debiasing with debias set to False
- Adversarial Debiasing with debias set to True with number of epochs = 20
- Adversarial Debiasing with debias set to True with number of epochs = 50

Below, we've implemented the training for the plain classifier for you. 

**Hint:** Introduce a seed value to make `predict` repeatable.

In [ ]:
tf.reset_default_graph()
sess = tf.Session()
tf.disable_eager_execution()
plain_model = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='plain_classifier',
                          debias=False,
                          seed=1,
                          sess=sess)
plain_model.fit(dataset_orig_train)

In [ ]:
### YOUR CODE HERE
#Adversarial Debiasing algorithm (variation 1)
#debiased_model_1 = 
tf.reset_default_graph()
sess = tf.Session()
tf.disable_eager_execution()
debiased_model_1 = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='debiasing_classifier',
                          num_epochs=20,
                          debias=True,
                          seed = 1,
                          sess=sess)
debiased_model_1.fit(dataset_orig_train)

In [ ]:
### YOUR CODE HERE
#Adversarial Debiasing algorithm (variation 2)
#debiased_model_2 = 
tf.reset_default_graph()
sess = tf.Session()
tf.disable_eager_execution()
debiased_model_2 = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='debiasing_classifier',
                          num_epochs=50,
                          debias=True,
                          seed=1,
                          sess=sess)
debiased_model_2.fit(dataset_orig_train)

Let's now predict on the test dataset and measure the fairness of the predictions.

In [ ]:
# Apply the plain model to test data
plain_model_test = plain_model.predict(dataset_orig_test)
debiased_model_1_test = debiased_model_1.predict(dataset_orig_test)
debiased_model_2_test = debiased_model_2.predict(dataset_orig_test)

In [ ]:
# Metrics for the dataset from plain model (without debiasing)
display(Markdown("#### Plain model - without debiasing - classification metrics"))

classified_metric_nodebiasing_test = ClassificationMetric(dataset_orig_test, 
                                                 plain_model_test,
                                                 unprivileged_groups=unprivileged_groups,
                                                 privileged_groups=privileged_groups)
print("Test set: Classification accuracy = %f" % classified_metric_nodebiasing_test.accuracy())
TPR = classified_metric_nodebiasing_test.true_positive_rate()
TNR = classified_metric_nodebiasing_test.true_negative_rate()
bal_acc_nodebiasing_test = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = %f" % bal_acc_nodebiasing_test)
print("Test set: Disparate impact = %f" % classified_metric_nodebiasing_test.disparate_impact())
print("Test set: Equal opportunity difference = %f" % classified_metric_nodebiasing_test.equal_opportunity_difference())
print("Test set: Average odds difference = %f" % classified_metric_nodebiasing_test.average_odds_difference())
print("Test set: Theil_index = %f" % classified_metric_nodebiasing_test.theil_index())

# Metrics for the dataset from model with debiasing (1)
display(Markdown("#### Model 1 - with debiasing - classification metrics"))
classified_metric_debiasing_test = ClassificationMetric(dataset_orig_test, 
                                                 debiased_model_1_test,
                                                 unprivileged_groups=unprivileged_groups,
                                                 privileged_groups=privileged_groups)
print("Test set: Classification accuracy = %f" % classified_metric_debiasing_test.accuracy())
TPR = classified_metric_debiasing_test.true_positive_rate()
TNR = classified_metric_debiasing_test.true_negative_rate()
bal_acc_debiasing_test = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = %f" % bal_acc_debiasing_test)
print("Test set: Disparate impact = %f" % classified_metric_debiasing_test.disparate_impact())
print("Test set: Equal opportunity difference = %f" % classified_metric_debiasing_test.equal_opportunity_difference())
print("Test set: Average odds difference = %f" % classified_metric_debiasing_test.average_odds_difference())
print("Test set: Theil_index = %f" % classified_metric_debiasing_test.theil_index())

# Metrics for the dataset from model with debiasing (2)
display(Markdown("#### Model 2 - with debiasing - classification metrics"))
classified_metric_debiasing_test = ClassificationMetric(dataset_orig_test, 
                                                 debiased_model_2_test,
                                                 unprivileged_groups=unprivileged_groups,
                                                 privileged_groups=privileged_groups)
print("Test set: Classification accuracy = %f" % classified_metric_debiasing_test.accuracy())
TPR = classified_metric_debiasing_test.true_positive_rate()
TNR = classified_metric_debiasing_test.true_negative_rate()
bal_acc_debiasing_test = 0.5*(TPR+TNR)
print("Test set: Balanced classification accuracy = %f" % bal_acc_debiasing_test)
print("Test set: Disparate impact = %f" % classified_metric_debiasing_test.disparate_impact())
print("Test set: Equal opportunity difference = %f" % classified_metric_debiasing_test.equal_opportunity_difference())
print("Test set: Average odds difference = %f" % classified_metric_debiasing_test.average_odds_difference())
print("Test set: Theil_index = %f" % classified_metric_debiasing_test.theil_index())

Select 3 fairness metrics and interpret their results. Did your choice of parameters change the results of the fairness metrics?

*Interpretation:*

For my interpretation, I selected Disparate Impact, Equal Opportunity Difference, and Average Odds Difference.

**Equal opportunity difference :** For the plain model, the metric is less than 0, indicating higher benefit for the privileged group  - men. For model 1 when the model is trained with 20 epochs, we see the value is greater than 0, indicating higher benefit for the unprivileged group - women. For model 2, we see the model is indicating higher benefit for the privileged group compared to the plain model! To summarize, model 1 is demonstrating stronger bias mitigation when it comes to this metric.

**Disparate impact:** The ideal value for this metric is a 1.0. 
For the plain model, the metric is close to but less than 1, indiacting a higher benefit for the privileged group, men. For model 1, the metric is over 1, indicating a higher benefit for women. For model 2, we again see this model is showing higher benefit for men compared to even the plain classifier due to the low values of the metric.

**Theil_index:** A value of 0 implies perfect fairness. Model 1 is reporting the closest scores to these values.

Summary: For the 3 metrics we looked at, it appears that training the model with 50 epochs (variation 2) resulted in increases in the bias metrics in certain cases. Variation 1, or training the model on 20 epochs, seemed to succesfully help reduce the bias in relation to the 3 metrics we selected. 

Notice how the choice of epochs could potentially change the result in the fairness metrics, which helps us identify that testing and evaluating design decisions during the AI development process and their impact on AI Ethics metrics, including fairness metrics, is important.

If we did not use a seed, the immediate next step would be to include re-running the analysis to construct a confidence interval of the metric values.

-----------------------------------------------------------------